In [ ]:

import pandas as pd
from pathlib import Path

# 1. Dossiers des données
dossier = Path("../data/totalité_juillet_septembre_2025")

# 2. Charger les 16 pluviomètres déjà nettoyés et fusionnés
df = pd.read_csv(
    dossier / "pluviometres_16_fusionnes.csv",
    parse_dates=["date"]
)

# 3. Ajouter les stations d'oxygène dissous
for station in ["Q01", "Q02", "Q03", "Q04"]:

    fichier = dossier / "totalité_juillet_septembre_2025" / f"{station}.csv"

    if not fichier.exists():
        print(f"Attention : {station}.csv introuvable")
        continue

    oxygene = pd.read_csv(
        fichier,
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    # Garder la date et la mesure d'oxygène
    oxygene = oxygene.iloc[:, :2].copy()
    oxygene.columns = ["date", station]

    # Convertir les dates et les mesures
    oxygene["date"] = pd.to_datetime(
        oxygene["date"].astype(str),
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )

    oxygene[station] = pd.to_numeric(
        oxygene[station],
        errors="coerce"
    )

    # Supprimer les dates invalides
    oxygene = oxygene.dropna(subset=["date"])

    # Éviter de multiplier les lignes en cas de doublons
    if oxygene["date"].duplicated().any():
        raise ValueError(f"Dates en double dans {station}.csv")

    # Fusionner selon la date
    df = df.merge(oxygene, on="date", how="outer")

# 4. Garder uniquement juillet, août et septembre 2025
df = df[
    (df["date"] >= "2025-07-01") &
    (df["date"] < "2025-10-01")
]

# 5. Trier chronologiquement
df = df.sort_values("date").reset_index(drop=True)

# 6. Vérifier le résultat
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))
print("\nValeurs manquantes :")
print(df.isna().sum())

display(df.head())


Problemes apparus: 
1. L'encodage : Python ne reconnaît pas certains caractères accentués.
2. Le séparateur : Python coupe le nom date/time (SER UTC+2,00) au niveau de la virgule, alors que les colonnes des fichers d'oxygène sont séparées par des tabulations.  

Correction:  
  
   df_temp = pd.read_csv(. 
    fichier,  
    sep="\t",  
    encoding="cp1252"
) 

Á savori aussi: dans les fichers des pluviometres il y a plusieurs mesures. 
 
- La première mesure de pluie est une donnée brute, non corrigée.
- Une valeur supérieure à 10 mm en 5 minutes correspond à un étalonnage du pluviomètre, et non à une pluie réelle.
- Les colonnes pré-validées ou validées doivent être privilégiées lorsqu'elles sont disponibles et que leur signification est confirmée.  

Dans ce code: choix d'utiliser la mesure pré-validée lorsqu'elle existe ; sinon utiliser la mesure brute, mais remplacer les valeurs supérieures à 10 mm par une valeur manquante (NaN), jamais par zéro.

#### Verification données

In [ ]:
# Compter les valeurs manquantes par station
valeurs_manquantes = df.isna().sum()

print("Valeurs manquantes par station :")
print(valeurs_manquantes)